# Comparing sets: UNION, INTERSECT, EXCEPT

Practice notebook for the datafromzero SQL lesson **Comparing sets: UNION, INTERSECT, EXCEPT**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. The two sets: A (ordered before April) and B (ordered at Harbor)

In [ ]:
con.sql("""
SELECT DISTINCT customer_id
FROM orders
WHERE ordered_at < DATE '2026-04-01' AND customer_id IS NOT NULL
ORDER BY customer_id
""").show()

In [ ]:
con.sql("""
SELECT DISTINCT customer_id
FROM orders
WHERE store_id = 3
ORDER BY customer_id
""").show()

### Step 2. INTERSECT: in both

In [ ]:
con.sql("""
SELECT customer_id FROM orders
WHERE ordered_at < DATE '2026-04-01' AND customer_id IS NOT NULL
INTERSECT
SELECT customer_id FROM orders
WHERE store_id = 3
ORDER BY customer_id
""").show()

### Step 3. EXCEPT, in both directions

In [ ]:
con.sql("""
SELECT customer_id FROM orders
WHERE ordered_at < DATE '2026-04-01' AND customer_id IS NOT NULL
EXCEPT
SELECT customer_id FROM orders
WHERE store_id = 3
ORDER BY customer_id
""").show()

In [ ]:
con.sql("""
SELECT customer_id FROM orders
WHERE store_id = 3
EXCEPT
SELECT customer_id FROM orders
WHERE ordered_at < DATE '2026-04-01' AND customer_id IS NOT NULL
ORDER BY customer_id
""").show()

### Step 4. UNION and UNION ALL

In [ ]:
con.sql("""
SELECT COUNT(*) AS union_rows
FROM (
  SELECT DISTINCT customer_id FROM orders
  WHERE ordered_at < DATE '2026-04-01' AND customer_id IS NOT NULL
  UNION
  SELECT DISTINCT customer_id FROM orders WHERE store_id = 3
)
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS union_all_rows
FROM (
  SELECT DISTINCT customer_id FROM orders
  WHERE ordered_at < DATE '2026-04-01' AND customer_id IS NOT NULL
  UNION ALL
  SELECT DISTINCT customer_id FROM orders WHERE store_id = 3
)
""").show()

### Step 5. Line up the columns: a total row

In [ ]:
con.sql("""
SELECT s.store_name, COUNT(*) AS orders
FROM orders o
JOIN stores s ON s.store_id = o.store_id
GROUP BY s.store_name
UNION ALL
SELECT 'All stores', COUNT(*)
FROM orders
ORDER BY orders
""").show()

### Step 6. Coverage: bought from every category

In [ ]:
con.sql("""
SELECT o.customer_id, COUNT(DISTINCT p.category) AS categories
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
JOIN products p ON p.product_id = oi.product_id
WHERE o.status = 'completed' AND o.customer_id IS NOT NULL
GROUP BY o.customer_id
HAVING COUNT(DISTINCT p.category) = (SELECT COUNT(DISTINCT category) FROM products)
ORDER BY o.customer_id
""").show()

### Step 7. Verify

In [ ]:
con.sql("""
WITH before_april AS (
  SELECT DISTINCT customer_id FROM orders
  WHERE ordered_at < DATE '2026-04-01' AND customer_id IS NOT NULL
),
at_harbor AS (
  SELECT DISTINCT customer_id FROM orders WHERE store_id = 3
)
SELECT (SELECT COUNT(*) FROM before_april) AS a,
       (SELECT COUNT(*) FROM at_harbor) AS b,
       (SELECT COUNT(*) FROM (SELECT * FROM before_april INTERSECT SELECT * FROM at_harbor)) AS in_both,
       (SELECT COUNT(*) FROM (SELECT * FROM before_april UNION SELECT * FROM at_harbor)) AS in_either
""").show()

In [ ]:
con.sql("""
SELECT categories, COUNT(*) AS customers
FROM (
  SELECT o.customer_id, COUNT(DISTINCT p.category) AS categories
  FROM orders o
  JOIN order_items oi ON oi.order_id = o.order_id
  JOIN products p ON p.product_id = oi.product_id
  WHERE o.status = 'completed' AND o.customer_id IS NOT NULL
  GROUP BY o.customer_id
) AS per_customer
GROUP BY categories
ORDER BY categories
""").show()

## Exercise 1. Customers who ordered at both Old Town and Campus

The first query is done for you. Add the operator and the second query, sorted by `customer_id`. Expected: 1 column.

In [ ]:
con.sql("""
SELECT customer_id FROM orders WHERE store_id = 1
-- add the operator and the second query here
ORDER BY customer_id
""").show()

## Exercise 2. Customers who ordered at all three cafes

Before you run anything, write one sentence in the cell below: what one row of the result is, and which customers must
not appear. Then use the coverage pattern of step 6, not `INTERSECT`. Count any order, canceled included, and keep the
walk-in order out.

In [ ]:
# My expected result, in words:
#

In [ ]:
# Query: customers whose orders touch every store

## Exercise 3. Spot the flaw

A colleague wants the customer and store pairs that appear both before and after April. Run the query, then find out what
went wrong and write the right one.

In [ ]:
con.sql("""
SELECT customer_id, store_id FROM orders WHERE ordered_at < DATE '2026-04-01'
INTERSECT
SELECT store_id, customer_id FROM orders WHERE ordered_at >= DATE '2026-04-01'
""").show()

In [ ]:
# Your corrected query: